# Probability of Fire (PoF) — Training The Model with Less Features

**Master:** Physics of Data \
**Course:** Laboratory of Computational Physics (LCP), Module B \
**Authors:** Gabriela Landinez Rangel, Andres Rojas Lozano, Fatemeh Dashti, Arash Taraz Jamshidi

*This notebook was created by us to present the final project for LCP MOD B, However it is based on public available code from the Probability of Fire project by ECMWF: https://ecmwf.github.io/AI-Probability-of-Fire/pof-trainer/. With this work, we aim to provide a clear, but friendly, guide with straightforward instructions and detailed explanations for students who, like us, want to reproduce this framework.*

In [1]:
import pandas as pd
import os
import time
import joblib
import matplotlib.pyplot as plt
import xgboost as xgb
from xgboost import XGBClassifier, plot_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay


In [2]:

df = pd.read_parquet("./data/training_data.parquet")

print("Loaded data shape:", df.shape)
print("Columns:", list(df.columns))

Loaded data shape: (216233, 14)
Columns: ['AF', 'PR', 'T2', 'RH', 'WS', 'FU_LL', 'FU_LW', 'FU_DF', 'FU_DW', 'DF', 'DW', 'LF', 'PO', 'RD']


| Model              | Features                     | Scientific question                  |
| ------------------ | ---------------------------- | ------------------------------------ |
| Full model         | all 13 features              | baseline                             |
| Weather-only       | `PR, T2, RH, WS`             | can weather alone explain fire risk? |
| Weather + moisture | `PR, T2, RH, WS, DF, DW, LF` | does fuel dryness improve the model? |
| Weather + human    | `PR, T2, RH, WS, PO, RD`     | do ignition proxies help?            |
| Moisture-only      | `DF, DW, LF`                 | how much comes from fuel moisture?   |


In [5]:
import pandas as pd
import joblib

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

df = pd.read_parquet("./data/training_data.parquet")

feature_sets = {
    "full": [
        "PR", "T2", "RH", "WS",
        "FU_LL", "FU_LW", "FU_DF", "FU_DW",
        "DF", "DW", "LF",
        "PO", "RD"
    ],

    "weather_only": [
        "PR", "T2", "RH", "WS"
    ],

    "weather_moisture": [
        "PR", "T2", "RH", "WS",
        "DF", "DW", "LF"
    ],

    "weather_human": [
        "PR", "T2", "RH", "WS",
        "PO", "RD"
    ],

    "moisture_only": [
        "DF", "DW", "LF"
    ]
}

results = []

for model_name, features in feature_sets.items():

    X = df[features]
    y = df["AF"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    model = XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.1,
        #subsample=0.8,
        #colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    auc = roc_auc_score(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)

    results.append({
        "model": model_name,
        "n_features": len(features),
        "ROC_AUC": auc,
        "Average_Precision": ap,
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1]
    })

    joblib.dump(model, f"./data/POF_model_{model_name}.joblib")

results_df = pd.DataFrame(results)
results_df

,model,n_features,ROC_AUC,Average_Precision,TN,FP,FN,TP
0,full,13,0.969480,0.196699,42234,699,155,159
1,weather_only,4,0.965956,0.167193,41653,1280,104,210
2,weather_moisture,7,0.967132,0.174505,42003,930,132,182
3,weather_human,6,0.968233,0.182709,41959,974,132,182
4,moisture_only,3,0.909876,0.079500,40033,2900,142,172


The feature-ablation experiment shows that the weather-only model already achieves high ROC-AUC, meaning that temperature, humidity, wind, and precipitation contain strong information about fire-prone conditions. However, the full model remains the most complete because it also includes fuel, moisture, population, and road-density information.

| Result                                                       | interpretation                                                |
| ------------------------------------------------------------ | ------------------------------------------------------ |
| `full` has highest ROC-AUC and Average Precision             | Best overall model                                     |
| `weather_only` is very close to full                         | Atmospheric conditions are very strong drivers         |
| `weather_moisture` is also close                             | Fuel dryness adds physically meaningful information    |
| `weather_human` has good ROC-AUC but lower Average Precision | Human proxies help spatially, but not enough alone     |
| `moisture_only` is clearly weaker                            | Moisture matters, but without weather it is not enough |


In [6]:
from pathlib import Path

print("Model files:")
for f in sorted(Path("data").glob("POF_model_*.joblib")):
    print(f.name)

print("\nFeature-list files:")
for f in sorted(Path("data").glob("POF_features_*.json")):
    print(f.name)

Model files:
POF_model_full.joblib
POF_model_moisture_only.joblib
POF_model_weather_human.joblib
POF_model_weather_moisture.joblib
POF_model_weather_only.joblib

Feature-list files:
POF_features_full.json
POF_features_moisture_only.json
POF_features_weather_human.json
POF_features_weather_moisture.json
POF_features_weather_only.json
